
# 02 — Fill non-trained baseline metrics for E-GEO benchmark

This notebook is a lightweight add-on for the manuscript pipeline. It does **not** retrain the 01 models and does **not** rerun the full pretrained-neural notebook. It only fills the missing non-trained baseline values that are useful for Supplementary Table S1:

- random ranking baseline
- lexical TF-IDF baseline
- optional E5-base pretrained embedding baseline on the official test split

The notebook writes a compact CSV that notebook 04 can read:

```text
data/egeo_02_nontrained_baseline_summary.csv
```

If an official processed test file is not available, the notebook still writes validation and cross-model rows and clearly marks official-test rows as unavailable.


In [ ]:

# Optional install for official-test E5 computation.
# Leave RUN_E5_OFFICIAL_TEST = False if you only want the fast lexical/random patch.
RUN_E5_OFFICIAL_TEST = True
E5_MODEL_NAME = "intfloat/e5-base-v2"
E5_BATCH_SIZE = 64
RANDOM_SEED = 42

# If sentence-transformers is missing and RUN_E5_OFFICIAL_TEST=True, this notebook will try to install it.


In [ ]:

from pathlib import Path
import sys, os, math, warnings, subprocess, importlib.util
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/Finance Research/E-GEO-ML")
if not PROJECT_ROOT.exists():
    CWD = Path.cwd().resolve()
    if (CWD / "data").exists() or (CWD / "raw_data").exists():
        PROJECT_ROOT = CWD
    elif (CWD.parent / "data").exists() or (CWD.parent / "raw_data").exists():
        PROJECT_ROOT = CWD.parent
    else:
        PROJECT_ROOT = CWD.parent if CWD.name.lower() == "code" else CWD

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
TABLE_DIR = DATA_DIR / "manuscript_tables_benchmark_final"
for d in [DATA_DIR, RESULTS_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEARCH_DIRS = [DATA_DIR, PROJECT_ROOT, TABLE_DIR, Path.cwd().resolve()]

def resolve_existing(name_or_path):
    p = Path(name_or_path)
    candidates = []
    if p.is_absolute():
        candidates.append(p)
    else:
        candidates.append(p)
        for d in SEARCH_DIRS:
            candidates.append(d / p.name)
            if len(p.parts) > 1:
                candidates.append(d / p)
    seen = set()
    for c in candidates:
        c = c.resolve()
        if c in seen:
            continue
        seen.add(c)
        if c.exists():
            return c
    return Path(name_or_path)

def read_csv_optional(name_or_path, **kwargs):
    p = resolve_existing(name_or_path)
    if p.exists():
        print("Loaded:", p)
        return pd.read_csv(p, **kwargs)
    print("Missing:", name_or_path)
    return None

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("TABLE_DIR:", TABLE_DIR)


## Helper functions

In [ ]:

from sklearn.metrics import average_precision_score, roc_auc_score, accuracy_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize


def ndcg_at_k_binary(labels, scores, k=3):
    labels = np.asarray(labels, dtype=float)
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores, kind="mergesort")
    rel = labels[order][:k]
    dcg = sum(rel[i] / math.log2(i + 2) for i in range(len(rel)))
    ideal = np.sort(labels)[::-1][:k]
    idcg = sum(ideal[i] / math.log2(i + 2) for i in range(len(ideal)))
    return dcg / idcg if idcg > 0 else np.nan


def query_level_metrics(eval_df, score_col="score", label_col="top3", k=3):
    rows = []
    for qid, g in eval_df.groupby("query_id"):
        g = g.copy()
        y = pd.to_numeric(g[label_col], errors="coerce").fillna(0).astype(int).to_numpy()
        s = pd.to_numeric(g[score_col], errors="coerce").fillna(-np.inf).to_numpy()
        order = np.argsort(-s, kind="mergesort")
        top = y[order][:k]
        positives = max(int(y.sum()), 1)
        rows.append({
            "query_id": qid,
            "precision@3": float(top.sum() / k),
            "recall@3": float(top.sum() / positives),
            "ndcg@3": float(ndcg_at_k_binary(y, s, k=k)),
        })
    return pd.DataFrame(rows)


def row_metrics(eval_df, score_col="score", label_col="top3"):
    y = pd.to_numeric(eval_df[label_col], errors="coerce").fillna(0).astype(int)
    s = pd.to_numeric(eval_df[score_col], errors="coerce").fillna(0.0)
    out = {
        "AUPRC": float(average_precision_score(y, s)) if y.nunique() > 1 else np.nan,
        "AUROC": float(roc_auc_score(y, s)) if y.nunique() > 1 else np.nan,
    }
    # F1/accuracy are not central for S1, but keeping them makes the file self-contained.
    if s.max() > s.min():
        threshold = float(np.quantile(s, 0.70))  # approximate top-30% positive rate threshold
    else:
        threshold = 0.5
    pred = (s >= threshold).astype(int)
    out.update({
        "Accuracy": float(accuracy_score(y, pred)),
        "F1": float(f1_score(y, pred, zero_division=0)),
        "threshold": threshold,
    })
    return out


def evaluate_scores(df, score_col, method_key, family, representative_method, split, label_model="gpt5", source="computed"):
    if df is None or len(df) == 0:
        return None
    needed = {"query_id", "top3", score_col}
    if not needed.issubset(df.columns):
        missing = sorted(needed - set(df.columns))
        print(f"Skipping {method_key} {split}: missing columns {missing}")
        return None
    q = query_level_metrics(df, score_col=score_col, label_col="top3", k=3)
    r = row_metrics(df, score_col=score_col, label_col="top3")
    out = {
        "family": family,
        "method_key": method_key,
        "representative_method": representative_method,
        "split": split,
        "label_model": label_model,
        "NDCG@3": float(q["ndcg@3"].mean()),
        "precision@3": float(q["precision@3"].mean()),
        "recall@3": float(q["recall@3"].mean()),
        "n_rows": int(len(df)),
        "n_queries": int(df["query_id"].nunique()),
        "source": source,
    }
    out.update(r)
    return out


def add_tfidf_query_product_cosine(target_df, fit_df=None, out_col="score_tfidf_compact_cosine"):
    df = target_df.copy()
    if out_col in df.columns:
        return df
    if "tfidf_query_product_cosine" in df.columns:
        df[out_col] = pd.to_numeric(df["tfidf_query_product_cosine"], errors="coerce").fillna(0.0)
        return df
    if fit_df is None:
        fit_df = df
    corpus = pd.concat([
        fit_df["query"].fillna("").astype(str),
        fit_df["product_text"].fillna("").astype(str)
    ], ignore_index=True)
    vectorizer = TfidfVectorizer(max_features=60000, ngram_range=(1, 2), min_df=2)
    vectorizer.fit(corpus)
    q = vectorizer.transform(df["query"].fillna("").astype(str))
    p = vectorizer.transform(df["product_text"].fillna("").astype(str))
    q = normalize(q)
    p = normalize(p)
    df[out_col] = np.asarray(q.multiply(p).sum(axis=1)).ravel()
    return df


def add_random_score(df, seed=42, out_col="score_random_ranking"):
    out = df.copy()
    rng = np.random.default_rng(seed)
    # Deterministic but product-row-specific enough for reproducible baseline.
    out[out_col] = rng.random(len(out))
    return out


## Load available processed data

In [ ]:

valid_split = read_csv_optional("egeo_gpt5_internal_valid_split.csv")
visibility_gpt5 = read_csv_optional("egeo_train_val_visibility_gpt5.csv")
rankings_long = read_csv_optional("egeo_train_val_rankings_long.csv")
products_flat = read_csv_optional("egeo_train_val_products_flat.csv")

# Official test files are produced by notebook 01 if raw_data/test_data.json and test ranking files were available.
test_gpt5 = read_csv_optional("egeo_test_visibility_gpt5.csv")
test_long = read_csv_optional("egeo_test_visibility_long_all_models.csv")

neural_query_all = read_csv_optional("egeo_pretrained_neural_query_metrics_all_label_models.csv")
neural_row_all = read_csv_optional("egeo_pretrained_neural_row_metrics_all_label_models.csv")

if valid_split is not None:
    print("Validation rows:", valid_split.shape, "queries:", valid_split["query_id"].nunique())
if test_gpt5 is not None:
    print("Official GPT-5 test rows:", test_gpt5.shape, "queries:", test_gpt5["query_id"].nunique())
else:
    print("No egeo_test_visibility_gpt5.csv found. Official-test non-trained baselines cannot be fully computed in this notebook until that file is available.")


## Rebuild cross-model validation long table only if needed

In [ ]:

# For random cross-model validation, try to use egeo_train_val_visibility_long_all_models.csv.
# If it was not saved/uploaded, rebuild the minimal long table by merging product rows with ranking labels.
train_val_long = read_csv_optional("egeo_train_val_visibility_long_all_models.csv")
if train_val_long is None and products_flat is not None and rankings_long is not None:
    print("Rebuilding minimal train_val long table from products_flat + rankings_long...")
    key_cols = [c for c in ["query_id", "custom_id", "product_num", "product_id"] if c in products_flat.columns and c in rankings_long.columns]
    if len(key_cols) >= 2:
        train_val_long = products_flat.merge(rankings_long, on=key_cols, how="inner", suffixes=("", "_rank"))
        print("Rebuilt train_val_long:", train_val_long.shape)
    else:
        print("Could not rebuild long table: shared key columns are insufficient:", key_cols)


## Compute fast random and lexical baselines

In [ ]:

rows = []

# Validation random baseline for GPT-5
if valid_split is not None:
    tmp = add_random_score(valid_split, seed=RANDOM_SEED)
    rows.append(evaluate_scores(
        tmp, "score_random_ranking", "random_ranking", "Random", "Random ranking", "validation", "gpt5", "02_computed"
    ))

# Cross-model random baseline on train/validation labels
if train_val_long is not None and {"model", "query_id", "top3"}.issubset(train_val_long.columns):
    for label_model, d in train_val_long.groupby("model"):
        d = add_random_score(d, seed=RANDOM_SEED)
        rows.append(evaluate_scores(
            d, "score_random_ranking", "random_ranking", "Random", "Random ranking", "validation", str(label_model), "02_computed_cross_model"
        ))

# Official test random baseline for GPT-5
if test_gpt5 is not None:
    tmp = add_random_score(test_gpt5, seed=RANDOM_SEED)
    rows.append(evaluate_scores(
        tmp, "score_random_ranking", "random_ranking", "Random", "Random ranking", "official_test", "gpt5", "02_computed"
    ))

# Validation lexical baseline: prefer already computed neural/lexical table; fallback to valid split cosine.
if neural_query_all is not None and neural_row_all is not None:
    q = neural_query_all[(neural_query_all["label_model"] == "gpt5") & (neural_query_all["score_model"] == "tfidf_compact_cosine")]
    r = neural_row_all[(neural_row_all["label_model"] == "gpt5") & (neural_row_all["score_model"] == "tfidf_compact_cosine")]
    if len(q) > 0:
        rows.append({
            "family": "Lexical", "method_key": "tfidf_compact_cosine", "representative_method": "TF-IDF cosine",
            "split": "validation", "label_model": "gpt5",
            "NDCG@3": float(q.iloc[0]["ndcg@3"]),
            "precision@3": float(q.iloc[0].get("precision@3", np.nan)),
            "recall@3": float(q.iloc[0].get("recall@3", np.nan)),
            "AUPRC": float(r.iloc[0]["AUPRC"]) if len(r) > 0 and "AUPRC" in r.columns else np.nan,
            "AUROC": float(r.iloc[0]["AUROC"]) if len(r) > 0 and "AUROC" in r.columns else np.nan,
            "Accuracy": float(r.iloc[0]["Accuracy"]) if len(r) > 0 and "Accuracy" in r.columns else np.nan,
            "F1": float(r.iloc[0]["F1"]) if len(r) > 0 and "F1" in r.columns else np.nan,
            "threshold": float(r.iloc[0]["threshold"]) if len(r) > 0 and "threshold" in r.columns else np.nan,
            "n_rows": int(q.iloc[0].get("n_rows", np.nan)) if pd.notna(q.iloc[0].get("n_rows", np.nan)) else np.nan,
            "n_queries": int(q.iloc[0].get("n_queries", np.nan)) if pd.notna(q.iloc[0].get("n_queries", np.nan)) else np.nan,
            "source": "precomputed_neural_table",
        })
else:
    if valid_split is not None:
        tmp = add_tfidf_query_product_cosine(valid_split, fit_df=visibility_gpt5 if visibility_gpt5 is not None else valid_split)
        rows.append(evaluate_scores(
            tmp, "score_tfidf_compact_cosine", "tfidf_compact_cosine", "Lexical", "TF-IDF cosine", "validation", "gpt5", "02_computed"
        ))

# Cross-model lexical/pretrained rows from precomputed neural table.
if neural_query_all is not None and neural_row_all is not None:
    merged = neural_query_all.merge(
        neural_row_all,
        on=["label_model", "score_model", "score_col", "n_rows", "n_queries"],
        how="left",
        suffixes=("_query", "_row")
    )
    for _, x in merged.iterrows():
        fam = "Lexical" if x["score_model"] == "tfidf_compact_cosine" else "Pretrained neural"
        rep = "TF-IDF cosine" if x["score_model"] == "tfidf_compact_cosine" else str(x["score_model"]).replace("embed_e5_base_cosine", "E5-base").replace("_", " ")
        rows.append({
            "family": fam,
            "method_key": x["score_model"],
            "representative_method": rep,
            "split": "validation",
            "label_model": x["label_model"],
            "NDCG@3": float(x["ndcg@3"]),
            "precision@3": float(x.get("precision@3", np.nan)),
            "recall@3": float(x.get("recall@3", np.nan)),
            "AUPRC": float(x.get("AUPRC", np.nan)),
            "AUROC": float(x.get("AUROC", np.nan)),
            "Accuracy": float(x.get("Accuracy", np.nan)),
            "F1": float(x.get("F1", np.nan)),
            "threshold": float(x.get("threshold", np.nan)),
            "n_rows": int(x.get("n_rows", np.nan)) if pd.notna(x.get("n_rows", np.nan)) else np.nan,
            "n_queries": int(x.get("n_queries", np.nan)) if pd.notna(x.get("n_queries", np.nan)) else np.nan,
            "source": "precomputed_neural_table",
        })

# Official test lexical baseline
if test_gpt5 is not None:
    fit_for_tfidf = visibility_gpt5 if visibility_gpt5 is not None else valid_split
    tmp = add_tfidf_query_product_cosine(test_gpt5, fit_df=fit_for_tfidf)
    rows.append(evaluate_scores(
        tmp, "score_tfidf_compact_cosine", "tfidf_compact_cosine", "Lexical", "TF-IDF cosine", "official_test", "gpt5", "02_computed"
    ))

rows = [r for r in rows if r is not None]
summary = pd.DataFrame(rows)
summary = summary.drop_duplicates(subset=["method_key", "split", "label_model"], keep="first")
print(summary.shape)
display(summary.head(20))


## Optional: compute E5-base baseline on official test

In [ ]:

if RUN_E5_OFFICIAL_TEST and test_gpt5 is not None:
    if importlib.util.find_spec("sentence_transformers") is None:
        print("Installing sentence-transformers for E5 official-test scoring...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import paired_cosine_distances

    print("Loading:", E5_MODEL_NAME)
    model = SentenceTransformer(E5_MODEL_NAME)

    d = test_gpt5.copy()
    queries = ("query: " + d["query"].fillna("").astype(str)).tolist()
    passages = ("passage: " + d["product_text"].fillna("").astype(str)).tolist()

    print("Encoding queries/passages:", len(d))
    q_emb = model.encode(queries, batch_size=E5_BATCH_SIZE, normalize_embeddings=True, show_progress_bar=True)
    p_emb = model.encode(passages, batch_size=E5_BATCH_SIZE, normalize_embeddings=True, show_progress_bar=True)
    d["score_embed_e5_base_cosine"] = np.sum(q_emb * p_emb, axis=1)

    e5_row = evaluate_scores(
        d,
        "score_embed_e5_base_cosine",
        "embed_e5_base_cosine",
        "Pretrained neural",
        "E5-base",
        "official_test",
        "gpt5",
        "02_computed_sentence_transformers"
    )
    if e5_row is not None:
        summary = pd.concat([summary, pd.DataFrame([e5_row])], ignore_index=True)
else:
    if not RUN_E5_OFFICIAL_TEST:
        print("RUN_E5_OFFICIAL_TEST=False, skipping official-test E5 computation.")
    elif test_gpt5 is None:
        print("No official test file, skipping official-test E5 computation.")


## Save 02 outputs

In [ ]:

# Standardize method labels.
METHOD_LABEL = {
    "random_ranking": "Random ranking",
    "tfidf_compact_cosine": "TF-IDF cosine",
    "embed_e5_base_cosine": "E5-base",
    "embed_minilm_l6_cosine": "MiniLM-L6 embedding",
    "embed_multiqa_minilm_cosine": "MultiQA MiniLM embedding",
    "rerank_msmarco_minilm_l6": "MS MARCO MiniLM-L6 reranker",
    "rerank_msmarco_minilm_l12": "MS MARCO MiniLM-L12 reranker",
    "rerank_bge_base": "BGE-base reranker",
    "hybrid_all_available": "Hybrid pretrained ensemble",
}
if len(summary) > 0:
    summary["representative_method"] = summary["method_key"].map(METHOD_LABEL).fillna(summary["representative_method"])
    sort_cols = ["family", "method_key", "split", "label_model"]
    summary = summary.sort_values(sort_cols).reset_index(drop=True)

summary_path = DATA_DIR / "egeo_02_nontrained_baseline_summary.csv"
summary.to_csv(summary_path, index=False)
print("Saved:", summary_path)

gpt5_view = summary[summary["label_model"].astype(str).eq("gpt5")].copy()
gpt5_path = DATA_DIR / "egeo_02_nontrained_baseline_gpt5_view.csv"
gpt5_view.to_csv(gpt5_path, index=False)
print("Saved:", gpt5_path)

cross_model_view = (
    summary[summary["split"].eq("validation")]
    .groupby(["family", "method_key", "representative_method"], as_index=False)
    .agg(cross_model_mean_ndcg3=("NDCG@3", "mean"), n_label_models=("label_model", "nunique"))
)
cross_path = DATA_DIR / "egeo_02_nontrained_cross_model_means.csv"
cross_model_view.to_csv(cross_path, index=False)
print("Saved:", cross_path)

display(gpt5_view[["family", "method_key", "split", "label_model", "NDCG@3", "AUPRC", "source"]].sort_values(["family", "method_key", "split"]))
display(cross_model_view)
